# NB08 — CNN Baseline: EfficientNet-B0 LOOCV

> **⚡ Run this notebook on Kaggle (GPU T4 / P100)**
> Upload the 26 patient folders as a Kaggle Dataset before running.

**Goal:** Train EfficientNet-B0 with Leave-One-Out Cross-Validation (LOOCV).
Soft label: GT image = 1.0, ±1 neighbors = 0.5, rest = 0.0.

**Reference:** DINOv2-Large Exp5 LOOCV: Top-1 23.1%, ±1 57.7%

**Output:** `cnn_scores.csv` → download and copy to `metrics_lab/outputs/` before running nb09.

## 0. Install Dependencies

In [ ]:
# ─── Kaggle / local install ──────────────────────────────────────────────────
import subprocess, sys

packages = [
    'pydicom',     # DICOM reading
    'numpy',
    'pandas',
    'matplotlib',
    'Pillow',
    'openpyxl',
    'timm',        # EfficientNet-B0 pretrained weights
    'scikit-learn',
    # torch & torchvision are pre-installed on Kaggle GPU kernels.
    # Uncomment the lines below only if running locally without torch:
    # 'torch',
    # 'torchvision',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies OK')

In [ ]:
import os
import numpy as np
import pandas as pd

# ─── Paths ─────────────────────────────────────────────────────────────────
# Option 1: Kaggle dataset mount
DATA_ROOT = '/kaggle/input/bssfp-freq-scout/1st_cohort_SAX'
GT_PATH   = '/kaggle/input/bssfp-freq-scout/1stcohortSAX (1).xlsx'
OUT_DIR   = '/kaggle/working'

# Option 2: Local run — comment out Option 1 and uncomment below
# DATA_ROOT = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
# GT_PATH   = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
# OUT_DIR   = 'outputs'

os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE = ['Patient-KP']
print(f'Config OK — OUT_DIR: {OUT_DIR}')

## 1. Data Loading

In [ ]:
import glob, re
import pydicom

def _sort_key(path):
    m = re.search(r'img(\d+)', os.path.basename(path), re.IGNORECASE)
    return int(m.group(1)) if m else 0

def load_patient_images(patient_dir, size=224):
    # Deduplicate to avoid Windows case-insensitive glob returning files twice
    dcm_set   = set(glob.glob(os.path.join(patient_dir, '*.dcm')) +
                    glob.glob(os.path.join(patient_dir, '*.DCM')))
    dcm_paths = sorted(dcm_set, key=_sort_key)
    images = []
    for p in dcm_paths:
        try:
            ds  = pydicom.dcmread(p, force=True)
            arr = ds.pixel_array.astype(np.float32)
            if arr.ndim == 3: arr = arr[0]
            if arr.max() > 0: arr /= arr.max()
            from PIL import Image
            img = Image.fromarray((arr * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
            images.append(np.asarray(img).astype(np.float32) / 255.0)
        except Exception as e:
            print(f'  [WARN] {p}: {e}')
    return images

def load_gt(xlsx_path):
    df = pd.read_excel(xlsx_path, header=None)
    header_row = next(i for i, row in df.iterrows() if any('Anonymized ID' in str(v) for v in row.values))
    df.columns = df.iloc[header_row].tolist()
    df = df.iloc[header_row+1:].reset_index(drop=True).dropna(subset=['Anonymized ID'])
    gt = {}
    for _, row in df.iterrows():
        pid  = str(row['Anonymized ID']).strip()
        ino  = int(row['image no'])
        n    = 7 if pid == 'Patient-KP' else 13
        step = 300.0 / (n - 1)
        gt[pid] = {'image_no': ino, 'offset_hz': -150 + (ino-1)*step, 'n_images': n}
    return gt

print('Loading all patients...')
all_patients = {}
for d in sorted(os.scandir(DATA_ROOT), key=lambda x: x.name):
    if not d.is_dir() or d.name in EXCLUDE: continue
    imgs = load_patient_images(d.path)
    if imgs:
        all_patients[d.name] = imgs
        print(f'  {d.name}: {len(imgs)} images')

gt_dict      = load_gt(GT_PATH)
patient_list = sorted(all_patients.keys())
print(f'Total: {len(patient_list)} patients')

## 2. Dataset, Model & Training

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

TRAIN_TF = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

VAL_TF = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FreqScoutDataset(Dataset):
    def __init__(self, patient_ids, all_patients, gt_dict, transform=None):
        self.samples   = []
        self.transform = transform
        for pid in patient_ids:
            gt_no = gt_dict[pid]['image_no']
            for i, img in enumerate(all_patients[pid]):
                diff  = abs((i+1) - gt_no)
                label = 1.0 if diff == 0 else (0.5 if diff == 1 else 0.0)
                img3  = (np.stack([img, img, img], axis=-1) * 255).astype(np.uint8)
                self.samples.append((img3, label, pid, i+1))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_u8, label, pid, img_no = self.samples[idx]
        img_t = self.transform(img_u8) if self.transform else \
            torch.from_numpy(img_u8.transpose(2,0,1).astype(np.float32)/255.0)
        return img_t, torch.tensor(label, dtype=torch.float32), pid, img_no

class EfficientNetB0Ranker(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        for p in self.backbone.parameters(): p.requires_grad = False
        for name, p in self.backbone.named_parameters():
            if any(t in name for t in ['blocks.5', 'blocks.6', 'conv_head', 'bn2']):
                p.requires_grad = True
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

# Quick sanity check
m = EfficientNetB0Ranker(pretrained=False)
print('Model OK. Output shape:', m(torch.randn(2,3,224,224)).shape)
n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Trainable params: {n_train:,}')

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_fold(train_pids, test_pid, all_patients, gt_dict,
                   epochs=30, batch_size=16, lr=1e-4, device=DEVICE):
    train_ds = FreqScoutDataset(train_pids, all_patients, gt_dict, transform=TRAIN_TF)
    val_ds   = FreqScoutDataset([test_pid], all_patients, gt_dict, transform=VAL_TF)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=13,          shuffle=False, num_workers=2, pin_memory=True)

    model     = EfficientNetB0Ranker(pretrained=True).to(device)
    optimiser = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimiser, T_max=epochs)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        for imgs, labels, _, __ in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            optimiser.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimiser.step()
        scheduler.step()
        if (epoch+1) % 10 == 0:
            print(f'  [{test_pid}] Epoch {epoch+1}/{epochs}')

    model.eval()
    results = []
    with torch.no_grad():
        for imgs, _, pid_batch, img_nos in val_dl:
            scores = model(imgs.to(device)).cpu().numpy()
            for score, pid, img_no in zip(scores, pid_batch, img_nos.tolist()):
                n    = gt_dict[pid]['n_images']
                step = 300.0 / (n-1)
                hz   = -150.0 + (img_no-1)*step
                results.append({'patient_id': pid, 'image_no': int(img_no),
                                'offset_hz': hz, 'cnn_score': float(score)})
    return results

print('Training function defined.')

## 3. Run LOOCV & Evaluate

In [ ]:
EPOCHS = 30; BATCH_SIZE = 16; LR = 1e-4
all_results = []

for fold_i, test_pid in enumerate(patient_list):
    train_pids = [p for p in patient_list if p != test_pid]
    print(f'\n[Fold {fold_i+1}/{len(patient_list)}] Test: {test_pid}')
    fold_results = train_one_fold(train_pids, test_pid, all_patients, gt_dict,
                                  epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR)
    all_results.extend(fold_results)

    scores     = [r['cnn_score'] for r in fold_results]
    pred_img   = fold_results[int(np.argmax(scores))]['image_no']
    gt_no      = gt_dict[test_pid]['image_no']
    marker     = '✅' if pred_img == gt_no else ('~' if abs(pred_img-gt_no)<=1 else '❌')
    print(f'  Pred: #{pred_img}  GT: #{gt_no}  {marker}')

df_cnn = pd.DataFrame(all_results)
csv_path = os.path.join(OUT_DIR, 'cnn_scores.csv')
df_cnn.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Evaluate
correct_count = pm1_count = mae_sum = n_valid = 0
for pid in patient_list:
    sub = df_cnn[df_cnn['patient_id']==pid].sort_values('image_no')
    if sub.empty: continue
    pred_no = int(sub.iloc[int(np.argmax(sub['cnn_score'].values))]['image_no'])
    gt_no   = gt_dict[pid]['image_no']
    n_imgs  = gt_dict[pid]['n_images']
    step    = 300.0 / (n_imgs - 1)
    mae_hz  = abs(pred_no - gt_no) * step
    correct_count += (pred_no == gt_no)
    pm1_count     += (abs(pred_no - gt_no) <= 1)
    mae_sum       += mae_hz
    n_valid       += 1

print('=' * 55)
print(f'  EfficientNet-B0 LOOCV  (n={n_valid})')
print(f'  Top-1 : {correct_count}/{n_valid} = {100*correct_count/n_valid:.1f}%')
print(f'  ±1    : {pm1_count}/{n_valid} = {100*pm1_count/n_valid:.1f}%')
print(f'  MAE   : {mae_sum/n_valid:.1f} Hz')
print('=' * 55)
print('\nReference: DINOv2-Large Exp5 -> Top-1 23.1%, +/-1 57.7%')
print('\nDownload cnn_scores.csv from Kaggle and place in metrics_lab/outputs/')
print('Then run nb09_multimetric_scoring.ipynb')